# Experiment: GW Corner And Matrix Plot Distributions

Objective:
- Generate publication-style parameter distribution figures for the BNS and NSBH injection catalogs.
- Keep the 6D corner plot for each population.
- Add a 3x3 matrix figure for each population with six 1D histograms and three 2D density panels.


In [ ]:
from pathlib import Path

import corner
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE = Path("<BASE_DIR>/gw-kn-multimodal")
BNS_CSV = BASE / "dataset/O5_sim_bns_aug/injections_final.csv"
NSBH_CSV = BASE / "dataset/O5_sim_nsbh_train/injections_final.csv"
OUTDIR = BASE / "figures" / "gw_params_plots"
OUTDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 13,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.dpi": 150,
})

BNS_CSV, NSBH_CSV, OUTDIR

## Plan

- Load the BNS and NSBH injection catalogs.
- Build standardized parameter frames for the two populations.
- Use `corner.corner()` for the full 6D distribution view.
- Use a custom `3x3` matrix plot for the compact summary view.


In [ ]:
bns = pd.read_csv(BNS_CSV)
nsbh = pd.read_csv(NSBH_CSV)

pd.DataFrame([
    {"dataset": "BNS", "rows": len(bns), "columns": len(bns.columns)},
    {"dataset": "NSBH", "rows": len(nsbh), "columns": len(nsbh.columns)},
])

In [ ]:
def plot_corner(frame, label_map, title, outpath, color):
    labels = [label_map[col] for col in frame.columns]
    fig = corner.corner(
        frame.to_numpy(),
        labels=labels,
        bins=35,
        color=color,
        smooth=1.0,
        fill_contours=True,
        plot_datapoints=False,
        show_titles=False,
        title_fmt=".2f",
        quantiles=[0.16, 0.5, 0.84],
        label_kwargs={"fontsize": 16},
        title_kwargs={"fontsize": 12},
        hist_kwargs={"density": True, "alpha": 0.9},
    )
    fig.suptitle(title, fontsize=18, y=0.995)
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_matrix_grid(frame, label_map, title, outpath, color, density_pairs, cmap):
    fig, axes = plt.subplots(3, 3, figsize=(15, 13))
    hist_columns = list(frame.columns)

    for ax, key in zip(axes[:2].flat, hist_columns):
        values = frame[key].to_numpy()
        ax.hist(values, bins=35, color=color, alpha=0.85, edgecolor="white", linewidth=0.7)
        ax.set_xlabel(label_map[key])
        ax.set_ylabel("Count")
        ax.set_title("")
        ax.grid(alpha=0.2, linestyle=":")

    for ax, (x_key, y_key, panel_title) in zip(axes[2], density_pairs):
        ax.hexbin(
            frame[x_key].to_numpy(),
            frame[y_key].to_numpy(),
            gridsize=45,
            mincnt=1,
            bins="log",
            cmap=cmap,
            linewidths=0,
        )
        ax.set_xlabel(label_map[x_key])
        ax.set_ylabel(label_map[y_key])
        ax.set_title(panel_title)

    fig.suptitle(title, fontsize=18, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.975])
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)


def build_bns_frame(df):
    frame = df[["mass1", "mass2", "spin1z", "spin2z", "distance", "inclination"]].dropna().rename(
        columns={
            "mass1": "m1",
            "mass2": "m2",
            "spin1z": "chi1",
            "spin2z": "chi2",
            "distance": "dL",
            "inclination": "inclination",
        }
    )
    label_map = {
        "m1": r"$m_1\,[M_\odot]$",
        "m2": r"$m_2\,[M_\odot]$",
        "chi1": r"$\chi_1$",
        "chi2": r"$\chi_2$",
        "dL": r"$d_L\,[\mathrm{Mpc}]$",
        "inclination": r"$\iota\,[\mathrm{rad}]$",
    }
    density_pairs = [
        ("m1", "m2", r"$m_1$ vs $m_2$"),
        ("m1", "chi1", r"$m_1$ vs $\chi_1$"),
        ("dL", "inclination", r"$d_L$ vs $\iota$"),
    ]
    return frame, label_map, density_pairs


def build_nsbh_frame(df):
    base = df[["mass1", "mass2", "spin1z", "spin2z", "distance", "inclination"]].dropna()
    mass1 = base["mass1"].to_numpy()
    mass2 = base["mass2"].to_numpy()
    spin1 = base["spin1z"].to_numpy()
    spin2 = base["spin2z"].to_numpy()

    frame = pd.DataFrame(
        {
            "m_ns": np.minimum(mass1, mass2),
            "m_bh": np.maximum(mass1, mass2),
            "chi_ns": np.where(mass1 < mass2, spin1, spin2),
            "chi_bh": np.where(mass1 >= mass2, spin1, spin2),
            "dL": base["distance"].to_numpy(),
            "inclination": base["inclination"].to_numpy(),
        }
    )
    label_map = {
        "m_ns": r"$m_{\mathrm{NS}}\,[M_\odot]$",
        "m_bh": r"$m_{\mathrm{BH}}\,[M_\odot]$",
        "chi_ns": r"$\chi_{\mathrm{NS}}$",
        "chi_bh": r"$\chi_{\mathrm{BH}}$",
        "dL": r"$d_L\,[\mathrm{Mpc}]$",
        "inclination": r"$\iota\,[\mathrm{rad}]$",
    }
    density_pairs = [
        ("m_ns", "m_bh", r"$m_{\mathrm{NS}}$ vs $m_{\mathrm{BH}}$"),
        ("m_bh", "chi_bh", r"$m_{\mathrm{BH}}$ vs $\chi_{\mathrm{BH}}$"),
        ("dL", "inclination", r"$d_L$ vs $\iota$"),
    ]
    return frame, label_map, density_pairs

## Build Standardized Parameter Frames

For NSBH, the lighter compact object is mapped to the NS axis and the heavier one to the BH axis on an event-by-event basis.


In [ ]:
bns_frame, bns_labels, bns_pairs = build_bns_frame(bns)
nsbh_frame, nsbh_labels, nsbh_pairs = build_nsbh_frame(nsbh)

{
    "bns_shape": bns_frame.shape,
    "nsbh_shape": nsbh_frame.shape,
    "bns_columns": list(bns_frame.columns),
    "nsbh_columns": list(nsbh_frame.columns),
}

## Generate Corner Plots


In [ ]:
bns_corner = OUTDIR / "bns_params_corner.png"
nsbh_corner = OUTDIR / "nsbh_params_corner.png"

plot_corner(bns_frame, bns_labels, "BNS Injection Parameters", bns_corner, "#1f77b4")
plot_corner(nsbh_frame, nsbh_labels, "NSBH Injection Parameters", nsbh_corner, "#d62728")

{
    "bns_corner": str(bns_corner),
    "nsbh_corner": str(nsbh_corner),
}

## Generate 3x3 Matrix Plots


In [ ]:
bns_matrix = OUTDIR / "bns_params_matrix.png"
nsbh_matrix = OUTDIR / "nsbh_params_matrix.png"

plot_matrix_grid(bns_frame, bns_labels, "Parameter distributions of BNS", bns_matrix, "#1f77b4", bns_pairs, "Blues")
plot_matrix_grid(nsbh_frame, nsbh_labels, "Parameter distributions of NSBH", nsbh_matrix, "#d62728", nsbh_pairs, "Reds")

{
    "bns_matrix": str(bns_matrix),
    "nsbh_matrix": str(nsbh_matrix),
}

In [ ]:
result = {
    "bns_rows_used": int(bns_frame.shape[0]),
    "nsbh_rows_used": int(nsbh_frame.shape[0]),
    "bns_corner": str(bns_corner),
    "nsbh_corner": str(nsbh_corner),
    "bns_matrix": str(bns_matrix),
    "nsbh_matrix": str(nsbh_matrix),
}
result

## Results

- The notebook now includes both the corner-plot method and the 3x3 matrix-plot method.
- Each population produces two figures: one corner plot and one matrix plot.
- The matrix plot uses the same parameter mapping and figure style as the current Python script.
